In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import requests

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing import image

In [14]:
# Load or create the sample dataset used for image extraction
sample_path = Path("../data/sample_5000.csv")

if sample_path.exists():
    sample_df = pd.read_csv(sample_path)
else:
    train_df = pd.read_csv("../data/student_resource/dataset/train.csv")
    sample_df = train_df.sample(n=5000, random_state=42).reset_index(drop=True)
    sample_df.to_csv(sample_path, index=False)

print("Sample dataset loaded")
print("Sample size:", len(sample_df))

Sample dataset loaded
Sample size: 5000


In [15]:
from pathlib import Path
import requests

image_dir = Path("../data/images")
image_dir.mkdir(parents=True, exist_ok=True)

for idx, url in enumerate(sample_df["image_link"]):
    img_path = image_dir / f"{idx}.jpg"

    if img_path.exists():
        continue

    try:
        response = requests.get(url, timeout=30)

        if response.status_code == 200:
            with open(img_path, "wb") as f:
                f.write(response.content)

    except Exception:
        continue

print("Images ready:", len(list(image_dir.glob("*.jpg"))))

Images ready: 5000


In [16]:
len(sample_df)

5000

In [17]:
# Load EfficientNetB0 for feature extraction
model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    pooling="avg"
)
print("EfficientNetB0 loaded")
print("Output shape:", model.output_shape)

EfficientNetB0 loaded
Output shape: (None, 1280)


In [18]:
image_dir = Path("../data/images")
batch_size = 32
all_features = []

for start in tqdm(range(0, len(sample_df), batch_size)):
    batch_imgs = []

    for idx in range(start, min(start + batch_size, len(sample_df))):
        img_path = image_dir / f"{idx}.jpg"
        if img_path.exists():
            img = image.load_img(img_path, target_size=(224, 224))
            img = image.img_to_array(img)
        else:
            img = np.zeros((224, 224, 3), dtype=np.float32)

        batch_imgs.append(img)

    batch_imgs = np.array(batch_imgs)
    batch_imgs = preprocess_input(batch_imgs)
    batch_features = model.predict(batch_imgs, verbose=0)
    all_features.append(batch_features)

image_features = np.vstack(all_features)
print("Image features shape:", image_features.shape)

  0%|          | 0/157 [00:00<?, ?it/s]

100%|██████████| 157/157 [03:42<00:00,  1.41s/it]

Image features shape: (5000, 1280)


In [20]:
import numpy as np

np.save("../data/image_features.npy", image_features)

print("Image features saved successfully!")

Image features saved successfully!


In [21]:
loaded = np.load("../data/image_features.npy")
print(loaded.shape)

(5000, 1280)
